# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzamanjatoi/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
!pip install -q huggingface_hub

from huggingface_hub import login
login()

In [11]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

def get_hf_token():
    token = userdata.get("HF_TOKEN")
    if token:
        return token

    return getpass.getpass(
        "Enter your Hugging Face READ token (hf_...): "
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

token = get_hf_token()

con.execute("SET VARIABLE hf_token = ?", [token])

con.execute("""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN getvariable('hf_token'))
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"{FACT}/month=2026-02/*.parquet"
MAR = f"{FACT}/month=2026-03/*.parquet"

print("Connected to FlyRank warehouse.")

Connected to FlyRank warehouse.


In [12]:
feb = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS impressions_feb,

        SUM(gsc_clicks)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS clicks_feb,

        SUM(gsc_sum_position)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS sum_position_feb,

        COUNT(*)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS measured_days_feb

    FROM read_parquet('{FEB}')

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING
        SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE) >= 100

        AND

        SUM(gsc_clicks)
            FILTER (WHERE gsc_data_available IS TRUE) >= 3
""").df()

feb["ctr_feb"] = (
    feb["clicks_feb"] /
    feb["impressions_feb"] * 100
)

feb["avg_position_feb"] = (
    feb["sum_position_feb"] /
    feb["impressions_feb"]
)

feb["position_tier"] = pd.cut(
    feb["avg_position_feb"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=[
        "top_3",
        "page_1",
        "striking",
        "page_3_5",
        "deep"
    ],
    include_lowest=True
)

print("February feature rows:", len(feb))

February feature rows: 29729


In [15]:
mar = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS impressions_mar,

        SUM(gsc_clicks)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS clicks_mar,

        COUNT(*)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS measured_days_mar

    FROM read_parquet('{MAR}')

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

mar = mar[
    mar["measured_days_mar"] > 0
].copy()

mar["impressions_mar"] = mar["impressions_mar"].fillna(0)
mar["clicks_mar"] = mar["clicks_mar"].fillna(0)

mar["went_dark"] = (
    mar["clicks_mar"] == 0
).astype(int)

In [16]:
frame = feb.merge(
    mar[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_mar",
            "clicks_mar",
            "measured_days_mar",
            "went_dark"
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

print("Final baseline frame:", frame.shape)
print("Future-label base rate:", frame["went_dark"].mean())

Final baseline frame: (29368, 13)
Future-label base rate: 0.03983928084990466


In [17]:
df = frame.copy()

assert df.shape[1] == 13, (
    f"expected 13 columns from ML-07, got {df.shape[1]}"
)

print(df.shape)
print(df["went_dark"].mean())
df.head()

(29368, 13)
0.03983928084990466


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,sum_position_feb,measured_days_feb,ctr_feb,avg_position_feb,position_tier,impressions_mar,clicks_mar,measured_days_mar,went_dark
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,29233.0,28,0.296443,28.886364,page_3_5,768.0,1.0,29,0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,27603.0,28,0.438048,17.273467,striking,3071.0,6.0,29,0
2,client_e547b89c05043229,content_c6dbda992ad84127,2868.0,3.0,51365.0,28,0.104603,17.909693,striking,3898.0,2.0,29,0
3,client_e547b89c05043229,content_fe83160838cdab99,2580.0,9.0,45826.0,28,0.348837,17.762016,striking,4255.0,8.0,29,0
4,client_e547b89c05043229,content_c5baa3a03cfccc41,4711.0,21.0,48200.0,28,0.445765,10.231373,striking,4095.0,9.0,29,0


In [18]:
print(df.shape)
print(df["went_dark"].mean())

(29368, 13)
0.03983928084990466


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import precision_score, recall_score, roc_auc_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [20]:
y = df["went_dark"]

## 1. Method choice and why

I'm using Logistic Regression for ML-08. My ML-07 baseline was a hand-written rule built from two binary triggers (negative CTR gap, sufficient volume) summed into a 0–3 score. Logistic regression is the natural next step: it estimates P(went_dark) from a linear combination of all my February signals at once instead of hand-picked thresholds, its coefficients stay interpretable (I can still say "a lower CTR gap moves the odds of going dark by X%," the same way I explained reason codes), it outputs a continuous probability I can rank by exactly like my baseline's descending score, and class_weight='balanced' handles the 4% positive rate without hand-tuning point values. I'm deliberately not jumping to Random Forest or Gradient Boosting yet — the assignment asks me to beat my baseline on the same footing first, and a linear model is the fairest, most explainable next step up.

In [22]:
import pandas as pd
import numpy as np

df = frame.copy()

assert df.shape[1] == 13, f"expected 13 columns from ML-07, got {df.shape[1]}"

print(df.shape)
print(df["went_dark"].mean())
df.head()

(29368, 13)
0.03983928084990466


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,sum_position_feb,measured_days_feb,ctr_feb,avg_position_feb,position_tier,impressions_mar,clicks_mar,measured_days_mar,went_dark
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,29233.0,28,0.296443,28.886364,page_3_5,768.0,1.0,29,0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,27603.0,28,0.438048,17.273467,striking,3071.0,6.0,29,0
2,client_e547b89c05043229,content_c6dbda992ad84127,2868.0,3.0,51365.0,28,0.104603,17.909693,striking,3898.0,2.0,29,0
3,client_e547b89c05043229,content_fe83160838cdab99,2580.0,9.0,45826.0,28,0.348837,17.762016,striking,4255.0,8.0,29,0
4,client_e547b89c05043229,content_c5baa3a03cfccc41,4711.0,21.0,48200.0,28,0.445765,10.231373,striking,4095.0,9.0,29,0


## 2. Split design

My rows are (client, content) pairs, and one client owns many pages. A random row split would let pages from the same client land in both train and test, letting the model partly learn "this client's pages behave a certain way" instead of the general Feb→Mar signal  that would inflate my score versus what my baseline was evaluated on. So I use a grouped split on client_hash_id (80/20), keeping every page from a given client on one side. I'm not doing a time-based split because I only have one feature period (Feb) and one outcome period (Mar) there's no earlier/later boundary within the data I have to split on further.

In [23]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print("client overlap:", len(overlap))  # must be 0

print("train:", train_df.shape, "went_dark rate:", train_df["went_dark"].mean())
print("test: ", test_df.shape,  "went_dark rate:", test_df["went_dark"].mean())

client overlap: 0
train: (16405, 13) went_dark rate: 0.04114599207558671
test:  (12963, 13) went_dark rate: 0.038185605183985186




```
# This is formatted as code
```

## 3. Train + compare vs my baseline

Same data (the ML-07 Feb-feature/Mar-outcome table), same metric (Precision@10/20/50 vs base rate), and — crucially — the same held-out test split for both methods, so the comparison is fair. I recompute my ML-07 rule on test_df rather than reusing the original full-population numbers, since the original 8%/4%/0%/0% figures were measured on all 29,368 rows, not this 20% split. I train logistic regression on train_df using log-scaled impressions, CTR, position, CTR gap, and one-hot position tier, then rank test_df by predicted probability the same way my rule ranked by score.

In [27]:
# Make sure ctr_gap exists in the ML-08 dataframe

tier_ctr = (
    df
    .groupby("position_tier", observed=True)["ctr_feb"]
    .median()
    .rename("position_tier_median_ctr")
)

df = df.join(tier_ctr, on="position_tier")

df["ctr_gap"] = (
    df["ctr_feb"] -
    df["position_tier_median_ctr"]
)

print(df.shape)
print(df[[
    "ctr_feb",
    "position_tier",
    "position_tier_median_ctr",
    "ctr_gap"
]].head())

(29368, 15)
    ctr_feb position_tier  position_tier_median_ctr   ctr_gap
0  0.296443      page_3_5                  0.209644  0.086799
1  0.438048      striking                  0.408094  0.029953
2  0.104603      striking                  0.408094 -0.303492
3  0.348837      striking                  0.408094 -0.059257
4  0.445765      striking                  0.408094  0.037671


In [28]:
from sklearn.model_selection import train_test_split

clients = df["client_hash_id"].drop_duplicates()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = df[df["client_hash_id"].isin(train_clients)].copy()
test_df = df[df["client_hash_id"].isin(test_clients)].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

Train shape: (23188, 15)
Test shape: (6180, 15)
Train clients: 24
Test clients: 6


In [29]:
print("Missing from train:")
print([
    col for col in [
        "impressions_feb",
        "ctr_feb",
        "avg_position_feb",
        "ctr_gap",
        "position_tier",
        "went_dark"
    ]
    if col not in train_df.columns
])

print("\nMissing from test:")
print([
    col for col in [
        "impressions_feb",
        "ctr_feb",
        "avg_position_feb",
        "ctr_gap",
        "position_tier",
        "went_dark"
    ]
    if col not in test_df.columns
])

Missing from train:
[]

Missing from test:
[]


In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_features = ["impressions_feb", "ctr_feb", "avg_position_feb", "ctr_gap"]
cat_features = ["position_tier"]

X_train = train_df[num_features + cat_features].copy()
X_test  = test_df[num_features + cat_features].copy()
y_train = train_df["went_dark"]
y_test  = test_df["went_dark"]

X_train["impressions_feb"] = np.log1p(X_train["impressions_feb"])
X_test["impressions_feb"]  = np.log1p(X_test["impressions_feb"])

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
logreg.fit(X_train, y_train)

test_df = test_df.copy()
test_df["pred_prob"] = logreg.predict_proba(X_test)[:, 1]

# Recompute the ML-07 rule on the SAME test split for a fair comparison
test_df["baseline_score"] = (
    2 * ((test_df["ctr_gap"] < 0) & (test_df["impressions_feb"] >= 100)).astype(int)
    + 1 * (test_df["impressions_feb"] >= 100).astype(int)
)

baseline_ranked = test_df.sort_values(
    ["baseline_score", "ctr_gap", "impressions_feb"], ascending=[False, True, False]
).reset_index(drop=True)
logreg_ranked = test_df.sort_values("pred_prob", ascending=False).reset_index(drop=True)

def precision_at_k(df_ranked, k, label_col="went_dark"):
    return df_ranked.head(k)[label_col].mean()

base_rate = test_df["went_dark"].mean()

comparison = pd.DataFrame({
    "metric": ["Precision@10", "Precision@20", "Precision@50", "Base rate (test split)"],
    "ML-07 baseline rule": [f"{precision_at_k(baseline_ranked, k):.1%}" for k in (10, 20, 50)] + [f"{base_rate:.1%}"],
    "ML-08 logistic regression": [f"{precision_at_k(logreg_ranked, k):.1%}" for k in (10, 20, 50)] + [f"{base_rate:.1%}"],
})
comparison

,metric,ML-07 baseline rule,ML-08 logistic regression
0,Precision@10,0.0%,0.0%
1,Precision@20,10.0%,15.0%
2,Precision@50,10.0%,18.0%
3,Base rate (test split),3.6%,3.6%


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [31]:
feature_names = (
    num_features
    + list(logreg.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(cat_features))
)
coefs = logreg.named_steps["clf"].coef_[0]
coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs}).sort_values("coef")
coef_df

,feature,coef
0,impressions_feb,-1.988216
6,position_tier_page_3_5,-1.141609
3,ctr_gap,-0.588160
7,position_tier_striking,-0.304189
1,ctr_feb,-0.271242
4,position_tier_deep,0.011479
5,position_tier_page_1,0.017582
2,avg_position_feb,0.291756
8,position_tier_top_3,0.324706


In [32]:
errors = test_df.copy()
threshold = errors["pred_prob"].quantile(1 - base_rate)  # flags roughly base_rate share, for inspection
errors["pred_label"] = (errors["pred_prob"] >= threshold).astype(int)

false_neg = errors[(errors["went_dark"] == 1) & (errors["pred_label"] == 0)]
false_pos = errors[(errors["went_dark"] == 0) & (errors["pred_label"] == 1)]

print("false negatives (went dark, model missed):", len(false_neg))
print("false positives (flagged, stayed alive):", len(false_pos))
false_neg[["impressions_feb", "ctr_feb", "avg_position_feb", "position_tier", "ctr_gap"]].describe()

false negatives (went dark, model missed): 197
false positives (flagged, stayed alive): 197


,impressions_feb,ctr_feb,avg_position_feb,ctr_gap
count,197.00000,197.000000,197.000000,197.000000
mean,3088.86802,0.452692,15.919290,0.120169
std,3481.20654,0.598263,10.428614,0.583900
min,142.00000,0.040606,1.460200,-0.367488
25%,1185.00000,0.119745,6.979354,-0.154930
50%,2137.00000,0.274123,15.037330,-0.035514
75%,4060.00000,0.502372,22.264095,0.162334
max,37771.00000,4.751620,69.695192,4.371753


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.